In [1]:
import logging
import pandas as pd
from pathlib import Path

from wildfire_susceptibility.config.loader import ConfigLoader
from wildfire_susceptibility.pipeline.preprocessor import WildfirePreprocessor
from wildfire_susceptibility.modeling.dataset_prep import DatasetPrep
from wildfire_susceptibility.modeling.train import ModelTrainer

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

CONFIG_PATH = "./wildfire_susceptibility/config/_working.yaml"  # or _working.yaml
cfg_obj = ConfigLoader.load(CONFIG_PATH)
cfg = cfg_obj.model_dump(mode="python")
cfg

c:\Users\rcorr\anaconda3\envs\wildfire\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'base': {'region': 'Essex',
  'boundary_shapefile': WindowsPath('data/silver/layers/boundary.shp'),
  'output_dir': WindowsPath('data/silver/layers'),
  'model_data_dir': WindowsPath('data/silver/model_datasets'),
  'figures_dir': WindowsPath('figures')},
 'processing': {'cell_size_m': 30.0,
  'crs': 'EPSG:27700',
  'training_years': (2009, 2020),
  'validation_years': (2020, 2022),
  'test_years': (2022, 9999),
  'force_recompute': False},
 'seasons': {'active': ['summer'],
  'seasonal_ndvi': True,
  'definitions': {'fall': [9, 10, 11],
   'spring': [3, 4, 5],
   'summer': [6, 7, 8],
   'winter': [12, 1, 2]}},
 'data_sources': {'cua': {'data_dir': WindowsPath('data/bronze/boundary')},
  'srtm': {'data_dir': WindowsPath('data/bronze/topography'),
   'tiles': ['n51_e000_1arc_v3.tif',
    'n51_e001_1arc_v3.tif',
    'n51_w001_1arc_v3.tif',
    'n52_e000_1arc_v3.tif',
    'n52_e001_1arc_v3.tif']},
  'haduk': {'data_dir': WindowsPath('data/bronze/climate_haduk'),
   'sources': ['tas', 'ta

In [2]:
RUN_PIPELINE = False  # set False if dataset_clean_<season>.csv already exist and force_recompute=False

dataset_paths = {}
if RUN_PIPELINE:
    wf = WildfirePreprocessor(CONFIG_PATH)
    dataset_paths = wf.run_full_pipeline()
else:
    for season in cfg["seasons"]["active"]:
        dataset_paths[season] = Path(cfg["base"]["model_data_dir"]) / f"dataset_clean_{season}.csv"

dataset_paths

{'summer': WindowsPath('data/silver/model_datasets/dataset_clean_summer.csv')}

In [5]:
dings = list(dataset_paths.items())
season, path = dings[0]
df = pd.read_csv(path)
df.dropna(inplace=False)

,elevation,slope,aspect,d_roads,d_rivers,d_activity,tas,tasmax,tasmin,rainfall,sfcWind,hurs,ndvi,label,_x,_y
8719,41.356953,2.902981,245.112410,0.123693,0.416773,0.000000,16.805685,22.480238,11.149773,54.068210,3.591560,75.204956,0.552599,0.0,584804.8,245314.2
8721,40.872540,1.917842,295.586820,0.067082,0.366197,0.000000,16.813967,22.487387,11.157903,54.034157,3.584522,75.199990,0.546333,0.0,584864.8,245314.2
8722,41.695404,3.547086,318.233860,0.042426,0.342053,0.000000,16.818110,22.490960,11.161967,54.017130,3.581002,75.197510,0.543156,0.0,584894.8,245314.2
8723,43.646618,4.332198,322.499540,0.030000,0.318904,0.000000,16.822250,22.494534,11.166033,54.000107,3.577483,75.195030,0.539950,0.0,584924.8,245314.2
8724,45.301437,4.026492,301.533720,0.000000,0.296985,0.000000,16.826391,22.498108,11.170097,53.983078,3.573964,75.192550,0.536714,0.0,584954.8,245314.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3850982,2.690375,3.053894,204.457100,0.000000,0.342053,0.300000,17.627554,22.314938,12.984325,49.646360,4.017493,75.759990,0.136646,0.0,579074.8,182104.2
3850983,0.981714,2.193394,220.342440,0.000000,0.318904,0.300000,17.629515,22.316760,12.986341,49.643550,4.017079,75.764150,0.132269,0.0,579104.8,182104.2
3850984,0.938834,0.243172,267.474060,0.000000,0.296985,0.283019,17.631477,22.318583,12.988356,49.640743,4.016664,75.768310,0.127893,0.0,579134.8,182104.2
3850985,0.970491,0.701309,274.777220,0.000000,0.276586,0.268328,17.633438,22.320404,12.990371,49.637940,4.016250,75.772480,0.123516,0.0,579164.8,182104.2


In [ ]:
all_results = {}

for season, path in dataset_paths.items():
    print(f"\n{'='*60}\n[{season}] loading dataset: {path}\n{'='*60}")
    df = pd.read_csv(path)

    prep = DatasetPrep(cfg)
    X_train, X_test, y_train, y_test = prep.stratified_split(df)

    groups_train = None
    if cfg["modeling"].get("cv_strategy") in ("spatial", "both"):
        groups_train = prep.assign_spatial_blocks(X_train)

    feature_cols = [c for c in X_train.columns if not c.startswith("_")]
    trainer = ModelTrainer(cfg)

    season_results = {}
    for model_name in cfg["modeling"]["models"]:
        print(f"\n--- [{season}] training {model_name} ---")

        def _on_trial(m_name, trial_number, trial_value):
            print(f"  [{season}][{m_name}] trial {trial_number+1}/{cfg['modeling']['optuna_n_trials']} "
                  f"AUC={trial_value:.4f}")

        result = trainer.train_one(
            season, model_name,
            X_train[feature_cols], y_train,
            X_test[feature_cols], y_test,
            groups_train=groups_train,
            progress_callback=_on_trial,
        )

        print(f"  >>> standard CV AUC={result['cv_auc_standard']:.4f}", end="")
        if result["cv_auc_spatial"] is not None:
            gap = result['cv_auc_standard'] - result['cv_auc_spatial']
            print(f" | spatial CV AUC={result['cv_auc_spatial']:.4f} | optimism gap={gap:.4f}", end="")
        print(f" | val AUC={result['val_auc']:.4f} | val F1={result['val_f1']:.4f}")

        season_results[model_name] = result

    all_results[season] = season_results

print("\nAll training complete.")

In [ ]:
rows = []
for season, models in all_results.items():
    for model_name, r in models.items():
        rows.append({
            "season": season,
            "model": model_name,
            "cv_auc_standard": r["cv_auc_standard"],
            "cv_auc_spatial": r["cv_auc_spatial"],
            "val_auc": r["val_auc"],
            "val_f1": r["val_f1"],
        })
pd.DataFrame(rows).sort_values(["season", "cv_auc_standard"], ascending=[True, False])

In [ ]:
from wildfire_susceptibility.reporting import generate_all

generate_all(config_path=CONFIG_PATH, seasons=list(dataset_paths.keys()), categories=["models", "eda"])